# C2 — Phân tích Màu sắc & SKU Bán chạy Q2/2026
**DATA EXPLORERS 2026 | Xe đạp Thống Nhất**

Phân tích và dự báo xu hướng màu sắc, mẫu xe (SKU) cho Q2/2026:
- Phân tích **Top màu** bán chạy theo doanh số và số lượng
- Phân tích **Top SKU** bán chạy và chậm bán
- Dự báo xu hướng **Top 5 màu** bằng SARIMAX cho T4–T6/2026
- Phát hiện SKU có **nguy cơ tăng trưởng âm**

**Thứ tự chạy:** Run All (tuần tự từ trên xuống)

## 1. Import Thư viện
> Nhập toàn bộ thư viện cần thiết. Chạy cell này đầu tiên.

In [ ]:
# ── Thư viện chuẩn ──────────────────────────────────────────────────────────
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ── Thống kê & mô hình ──────────────────────────────────────────────────────
import pmdarima as pm
from statsmodels.tsa.stattools import adfuller

print("✅ Import thư viện thành công")

## 2. Wrangle Data (Tải & Chuẩn hóa Dữ liệu)
> Tải dữ liệu giao dịch đầy đủ từ `fact_sales_full.csv`, chuẩn hóa kiểu dữ liệu và thêm các cột phái sinh (tháng, năm).
>
> **Về dữ liệu:** 22,357 đơn hàng, T1/2025–T3/2026. Một số dòng thiếu thông tin đại lý và sản phẩm — sẽ được xử lý ở bước Data Checks.

In [ ]:
# ── Tải dữ liệu giao dịch ────────────────────────────────────────────────────
df_full = pd.read_csv("data/fact_sales_full.csv", parse_dates=["order_date"])

# ── Thêm cột phái sinh ───────────────────────────────────────────────────────
df_full["month"]  = df_full["order_date"].dt.to_period("M")
df_full["year"]   = df_full["order_date"].dt.year

print(f"Shape            : {df_full.shape}")
print(f"Khoảng thời gian : {df_full.order_date.min().strftime('%m/%Y')} → {df_full.order_date.max().strftime('%m/%Y')}")
df_full.head()

## 3. Kiểm tra Dữ liệu (Data Checks)
> Kiểm tra giá trị thiếu, thống kê tổng quan, và làm sạch dữ liệu trước khi phân tích.

In [ ]:
# ── Kiểu dữ liệu & kích thước ────────────────────────────────────────────────
print(f"Shape: {df_full.shape}")
print(f"\nKiểu dữ liệu:")
print(df_full.dtypes)

In [ ]:
# ── Kiểm tra giá trị thiếu ───────────────────────────────────────────────────
missing_pct = df_full.isnull().mean() * 100
missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=False)
print("Tỷ lệ giá trị thiếu (%):")
print(missing_pct.round(2))
print(f"\n⚠️  Lưu ý: customer_name/province_name thiếu ~29% — do đơn hàng không gắn đại lý cụ thể")
print(f"⚠️  product_name/color/group_code thiếu ~2.1% — sẽ loại bỏ khi phân tích SKU")

In [ ]:
# ── Thống kê tổng quan ───────────────────────────────────────────────────────
print(f"Số đơn hàng    : {len(df_full):,}")
print(f"Số SKU (mẫu xe): {df_full.product_code.nunique():,}")
print(f"Số màu sắc     : {df_full.color.nunique():,}")
print(f"Số nhóm SP     : {df_full.group_code.nunique():,}")
print(f"Số tỉnh/thành  : {df_full.province_name.nunique():,}")
print(f"Tổng doanh thu : {df_full.line_total.sum()/1e9:.1f} tỷ VND")

In [ ]:
# ── Làm sạch: bỏ dòng thiếu thông tin sản phẩm ──────────────────────────────
df_clean = df_full.dropna(subset=["product_name", "color", "group_code"]).copy()
print(f"Sau clean: {len(df_clean):,} đơn ({len(df_clean)/len(df_full)*100:.1f}%)")
print(f"Đã loại  : {len(df_full)-len(df_clean):,} đơn thiếu thông tin sản phẩm")

## 4. Phân tích Khám phá Dữ liệu — Màu sắc (EDA)
> Phân tích top màu sắc bán chạy theo doanh số và số lượng. Xác định màu nào đang tăng trưởng.

In [ ]:
# ── Top 15 màu theo doanh số ─────────────────────────────────────────────────
top_colors = (
    df_clean.groupby("color")["line_total"]
    .sum()
    .sort_values(ascending=False)
    .head(15)
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Doanh số
ax1.barh(range(len(top_colors)), top_colors.values / 1e9, alpha=0.85)
ax1.set_yticks(range(len(top_colors)))
ax1.set_yticklabels(top_colors.index, fontsize=9)
ax1.set_xlabel("Doanh số (tỷ VND)")
ax1.set_title("Top 15 Màu — Doanh số", fontsize=12, fontweight="bold")
ax1.grid(axis="x", alpha=0.4)

# Số lượng
top_colors_qty = (
    df_clean.groupby("color")["quantity"]
    .sum()
    .sort_values(ascending=False)
    .head(15)
)
ax2.barh(range(len(top_colors_qty)), top_colors_qty.values, alpha=0.85)
ax2.set_yticks(range(len(top_colors_qty)))
ax2.set_yticklabels(top_colors_qty.index, fontsize=9)
ax2.set_xlabel("Số lượng (chiếc)")
ax2.set_title("Top 15 Màu — Số lượng", fontsize=12, fontweight="bold")
ax2.grid(axis="x", alpha=0.4)

fig.suptitle("Phân tích Màu sắc Bán chạy — Xe đạp Thống Nhất", fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()

In [ ]:
# ── Doanh số theo nhóm sản phẩm ─────────────────────────────────────────────
group_rev = (
    df_clean.groupby("group_name")["line_total"]
    .sum()
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(range(len(group_rev)), group_rev.values / 1e9, alpha=0.85)
ax.set_xticks(range(len(group_rev)))
ax.set_xticklabels(group_rev.index, rotation=20, ha="right", fontsize=10)
ax.set_ylabel("Doanh số (tỷ VND)")
ax.set_title("Doanh số theo Nhóm sản phẩm (T1/2025–T3/2026)", fontsize=12, fontweight="bold")
ax.grid(axis="y", alpha=0.4)
for i, v in enumerate(group_rev.values):
    ax.text(i, v/1e9 + 0.5, f"{v/1e9:.1f}T", ha="center", va="bottom", fontsize=9, fontweight="bold")
fig.tight_layout()
plt.show()

In [ ]:
# ── Xu hướng doanh số theo tháng × nhóm sản phẩm ────────────────────────────
pivot = (
    df_clean.groupby(["month", "group_name"])["line_total"]
    .sum()
    .unstack(fill_value=0)
    / 1e9
)
pivot.index = pivot.index.to_timestamp()

fig, ax = plt.subplots(figsize=(14, 6))
for col in pivot.columns:
    ax.plot(pivot.index, pivot[col], marker="o", linewidth=2, label=col)
ax.set_title("Xu hướng Doanh số theo Nhóm sản phẩm — Theo tháng", fontsize=12, fontweight="bold")
ax.set_xlabel("Tháng")
ax.set_ylabel("Doanh số (tỷ VND)")
ax.legend(loc="upper left", fontsize=9)
ax.grid(axis="y", alpha=0.4)
fig.tight_layout()
plt.show()

## 5. Phân tích Khám phá Dữ liệu — SKU (EDA)
> Xác định top SKU bán chạy và SKU chậm bán — thông tin quan trọng cho quyết định sản xuất và tồn kho.

In [ ]:
# ── Top 20 SKU bán chạy nhất ──────────────────────────────────────────────────
top_sku = (
    df_clean.groupby("product_name")["quantity"]
    .sum()
    .sort_values(ascending=False)
    .head(20)
)

fig, ax = plt.subplots(figsize=(12, 8))
ax.barh(range(len(top_sku)), top_sku.values, alpha=0.85)
ax.set_yticks(range(len(top_sku)))
ax.set_yticklabels([n[:45] for n in top_sku.index], fontsize=8)
ax.set_xlabel("Tổng số lượng bán (chiếc)")
ax.set_title("Top 20 SKU Bán chạy nhất (T1/2025–T3/2026)", fontsize=12, fontweight="bold")
ax.grid(axis="x", alpha=0.4)
fig.tight_layout()
plt.show()

In [ ]:
# ── SKU chậm bán (dưới phân vị 20%) ──────────────────────────────────────────
sku_avg = df_clean.groupby("product_name")["quantity"].mean()
slow_sku = sku_avg[sku_avg < sku_avg.quantile(0.20)].sort_values().head(20)

fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(range(len(slow_sku)), slow_sku.values, alpha=0.85)
ax.set_yticks(range(len(slow_sku)))
ax.set_yticklabels([n[:45] for n in slow_sku.index], fontsize=8)
ax.set_xlabel("Trung bình số lượng / đơn hàng")
ax.set_title("SKU Chậm bán — Dưới Phân vị 20%", fontsize=12, fontweight="bold")
ax.grid(axis="x", alpha=0.4)
fig.tight_layout()
plt.show()

## 6. Xây dựng Pipeline Dữ liệu — Chuẩn bị Chuỗi Thời gian
> Tổng hợp dữ liệu theo tháng × màu sắc để tạo chuỗi thời gian cho dự báo SARIMAX.
> Kiểm định ADF để xác định chuỗi có dừng không.

In [ ]:
# ── Chuẩn bị chuỗi thời gian theo màu ───────────────────────────────────────
top5_colors = top_colors.head(5).index.tolist()
print(f"Top 5 màu dự báo: {top5_colors}")

color_monthly = (
    df_clean[df_clean["color"].isin(top5_colors)]
    .groupby(["month", "color"])["quantity"]
    .sum()
    .unstack(fill_value=0)
)
color_monthly.index = color_monthly.index.to_timestamp()

print(f"\nShape chuỗi thời gian: {color_monthly.shape}")
color_monthly.tail()

In [ ]:
# ── Kiểm định ADF cho từng màu ───────────────────────────────────────────────
print("Kiểm định ADF — Tính dừng của chuỗi theo màu:")
print(f"{'Màu':<25} {'p-value':>10}  Kết luận")
print("-" * 50)
for col in top5_colors:
    pv = adfuller(color_monthly[col].dropna())[1]
    status = "✅ DỪNG" if pv < 0.05 else "⚠️  KHÔNG DỪNG"
    print(f"  {col:<23}: {pv:>8.4f}  {status}")

## 7. Mô hình SARIMAX — Dự báo Màu sắc Q2/2026
> Huấn luyện mô hình SARIMAX riêng cho từng màu top 5 bằng `auto_arima` (tự động chọn tham số tối ưu).
> Dự báo số lượng bán cho T4, T5, T6/2026.

In [ ]:
# ── SARIMAX dự báo 3 tháng Q2/2026 cho từng màu top 5 ───────────────────────
months_q2 = pd.date_range("2026-04-01", periods=3, freq="MS")
months_label = ["T4/2026", "T5/2026", "T6/2026"]
forecasts_color = {}

for col in top5_colors:
    try:
        model_c = pm.auto_arima(
            color_monthly[col].values,
            seasonal=True,
            m=3,
            stepwise=True,
            suppress_warnings=True,
            error_action="ignore",
        )
        pred = model_c.predict(n_periods=3)
        forecasts_color[col] = np.maximum(pred, 0)  # không để âm
        print(f"  ✅ {col:<20}: {[int(v) for v in forecasts_color[col]]}")
    except Exception as e:
        forecasts_color[col] = np.array([0, 0, 0])
        print(f"  ❌ {col:<20}: Lỗi — {e}")

print("\nDự báo hoàn tất.")

In [ ]:
# ── Biểu đồ dự báo màu sắc Q2/2026 ──────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Lịch sử
for col in top5_colors:
    ax1.plot(color_monthly.index, color_monthly[col], marker="o", linewidth=2, label=col)
ax1.set_title("Lịch sử Bán hàng — Top 5 Màu (chiếc/tháng)", fontsize=12, fontweight="bold")
ax1.set_xlabel("Tháng")
ax1.set_ylabel("Số lượng (chiếc)")
ax1.legend(fontsize=8)
ax1.grid(axis="y", alpha=0.4)

# Dự báo Q2
x = np.arange(3)
width = 0.15
for i, col in enumerate(top5_colors):
    ax2.bar(x + i * width, forecasts_color[col], width, label=col, alpha=0.85)
ax2.set_xticks(x + width * 2)
ax2.set_xticklabels(months_label)
ax2.set_title("Dự báo SARIMAX Q2/2026 — Top 5 Màu (chiếc)", fontsize=12, fontweight="bold")
ax2.set_ylabel("Số lượng dự báo (chiếc)")
ax2.legend(fontsize=8)
ax2.grid(axis="y", alpha=0.4)

fig.suptitle("Phân tích & Dự báo Xu hướng Màu sắc", fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()

## 8. Phát hiện SKU Rủi ro (Tăng trưởng Âm)
> So sánh doanh số 2 giai đoạn gần nhất để phát hiện SKU đang giảm dần — cần xem xét điều chỉnh sản xuất.

In [ ]:
# ── So sánh 2 giai đoạn: Q4/2025 vs Q3/2025 ─────────────────────────────────
recent = df_clean[df_clean["order_date"] >= "2025-10-01"].copy()
prev   = df_clean[(df_clean["order_date"] >= "2025-07-01") &
                   (df_clean["order_date"] <  "2025-10-01")].copy()

rev_recent = recent.groupby("product_name")["line_total"].sum().rename("recent")
rev_prev   = prev.groupby("product_name")["line_total"].sum().rename("prev")

growth = pd.concat([rev_recent, rev_prev], axis=1).fillna(0)
growth["growth_pct"] = ((growth["recent"] - growth["prev"]) / (growth["prev"] + 1)) * 100

# SKU tăng trưởng âm
slow = growth[growth["growth_pct"] < -20].sort_values("growth_pct")
print(f"Số SKU có tăng trưởng âm > 20%: {len(slow)}")

fig, ax = plt.subplots(figsize=(12, max(5, len(slow.head(20)) * 0.4)))
ax.barh(range(len(slow.head(20))), slow["growth_pct"].head(20), alpha=0.85)
ax.set_yticks(range(len(slow.head(20))))
ax.set_yticklabels([n[:40] for n in slow.head(20).index], fontsize=8)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Tăng trưởng doanh số (%)")
ax.set_title("SKU có Nguy cơ Tăng trưởng Âm (Q3→Q4/2025)", fontsize=12, fontweight="bold")
ax.grid(axis="x", alpha=0.4)
fig.tight_layout()
plt.show()

## 9. Xuất Kết quả
> Lưu dự báo màu sắc và danh sách SKU rủi ro vào thư mục `output/`.

In [ ]:
# ── Tổng hợp dự báo màu sắc ──────────────────────────────────────────────────
rows = []
for col, vals in forecasts_color.items():
    for month, val in zip(months_label, vals):
        rows.append({"color": col, "tháng": month, "qty_forecast": int(val)})

df_color_forecast = pd.DataFrame(rows)

# ── Lưu file ─────────────────────────────────────────────────────────────────
os.makedirs("output", exist_ok=True)
df_color_forecast.to_csv("output/predictions_color_sku.csv", index=False)
slow.reset_index().rename(columns={"product_name": "sku"}).to_csv("output/sku_risk_growth.csv", index=False)

print("✅ Đã lưu kết quả vào output/")
print("   → predictions_color_sku.csv")
print("   → sku_risk_growth.csv")
df_color_forecast

In [ ]:
# ── Tóm tắt kết quả cuối cùng ────────────────────────────────────────────────
print("=" * 55)
print("   KẾT QUẢ C2 — MÀU SẮC & SKU Q2/2026")
print("=" * 55)
print(f"\n→ Top 3 màu bán chạy (doanh số): {top_colors.head(3).index.tolist()}")
print(f"→ Top 3 SKU bán chạy           : {top_sku.head(3).index.tolist()}")
print(f"→ Số SKU chậm bán (<P20)       : {len(slow_sku)} SKU")
print(f"→ Số SKU nguy cơ tăng trưởng âm: {len(slow)} SKU")
print(f"\nDự báo Top 5 màu Q2/2026 (chiếc):")
for col in top5_colors:
    vals = [int(v) for v in forecasts_color[col]]
    print(f"  {col:<20}: T4={vals[0]:,}  T5={vals[1]:,}  T6={vals[2]:,}")
print("=" * 55)